# Proposed Model 3 — SAM2 + Global Re-acquisition

**Pipeline:** RGB video → click target → SAM2 local tracking → identity verification → full-frame deep re-detection after loss → bbox confirmation → reinitialize SAM2.


In [ ]:
import os
import sys
import subprocess
import urllib.request
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Colab: Runtime → Change runtime type → GPU.")

REPO = Path("/content/Target_Locker")
MODEL3_DIR = REPO / "proposed_model_3"
SAM_CKPT = Path("/content/sam2.1_hiera_tiny.pt")

if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/CptImtiaz/Target_Locker.git",
        str(REPO),
    ], check=True)
else:
    subprocess.run([
        "git", "-C", str(REPO),
        "pull", "--ff-only", "origin", "main",
    ], check=True)

requirements = MODEL3_DIR / "requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(f"requirements.txt not found: {requirements}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(requirements),
], check=True)

if not SAM_CKPT.exists():
    print("Downloading SAM2.1 Tiny checkpoint...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        str(SAM_CKPT),
    )

required_files = [
    MODEL3_DIR / "app_interface.py",
    MODEL3_DIR / "model3_tracker.py",
    MODEL3_DIR / "global_redetector.py",
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Required Model 3 file missing: {file_path}")

model3_path = str(MODEL3_DIR.resolve())
if model3_path not in sys.path:
    sys.path.insert(0, model3_path)

print("✅ Proposed Model 3 setup ready")
print("Pipeline: SAM2 + Deep Global Re-acquisition")
print("Checkpoint:", SAM_CKPT)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os
import sys
import runpy
from pathlib import Path

REPO = Path("/content/Target_Locker")
MODEL3_DIR = REPO / "proposed_model_3"

model3_path = str(MODEL3_DIR.resolve())
if model3_path not in sys.path:
    sys.path.insert(0, model3_path)

for module_name in ["model3_tracker", "global_redetector"]:
    sys.modules.pop(module_name, None)

os.chdir(MODEL3_DIR)

runpy.run_path(
    str(MODEL3_DIR / "app_interface.py"),
    run_name="__main__",
)
